***********Indian Word Game ZUDO**************

In [2]:
# ZUDO – AI WORD GAME  (Updated Graphics)

import tkinter as tk
from tkinter import messagebox
import random
import pandas as pd
import pygame
import os
import math

# ── Audio ─────────────────────────────────────────────────────
pygame.mixer.init()

try:
    pygame.mixer.music.load("sounds/bg.mpeg")
    pygame.mixer.music.set_volume(0.4)
    pygame.mixer.music.play(-1)
except Exception:
    pass

def load_sound(path, volume=0.8):
    try:
        s = pygame.mixer.Sound(path)
        s.set_volume(volume)
        return s
    except Exception:
        return None

click_sound   = load_sound("sounds/click.mp3",   volume=0.8)
warning_sound = load_sound("sounds/warning.mp3", volume=0.7)
wrong_sound   = load_sound("sounds/wrong.mp3",   volume=0.8)
win_sound     = load_sound("sounds/win.mp3",      volume=0.9)
coin_sound    = load_sound("sounds/coin.mp3",     volume=1.0)

def play_sound(sound):
    if sound:
        sound.play()

# ── Dataset loader ────────────────────────────────────────────
def load_round_data(round_num):
    path = f"round{round_num}_dataset.csv"
    if os.path.exists(path):
        return pd.read_csv(path)
    return pd.DataFrame(columns=["word", "category"])

# ── Global state ──────────────────────────────────────────────
players        = []
player_avatars = {}
scores         = {}

# ── Constants ─────────────────────────────────────────────────
MAX_PLAYERS = 4
ROUND_LIMIT = 3
TIME_LIMIT  = 20
WARNING_AT  = 5

WIN_W = 1100
WIN_H = 700

# ── Colours ───────────────────────────────────────────────────
BG_DEEP      = "#040410"
BG_PANEL     = "#0b0929"
BG_CARD      = "#100e35"
BORDER_DIM   = "#252260"
BORDER_VIVID = "#4f43c8"
PINK         = "#f472b6"
CYAN         = "#22d3ee"
YELLOW       = "#fde68a"
GREEN        = "#4ade80"
RED          = "#f87171"
PURPLE       = "#a78bfa"
WHITE        = "#f0eeff"
ORANGE       = "#fb923c"
GOLD         = "#fbbf24"

F_SMALL  = ("Courier New",  9)
F_BTN    = ("Courier New", 12, "bold")
F_TIMER  = ("Courier New", 52, "bold")
F_TITLE  = ("Courier New", 32, "bold")
F_HEAD   = ("Courier New", 16, "bold")

PLAYER_COLORS = [PINK, CYAN, GREEN, YELLOW]

BOY_AVATAR_PATH  = "boy.png"
GIRL_AVATAR_PATH = "girl.png"
COIN_IMAGE_PATH  = "coin.png"

# =============================================================
#  MAIN WINDOW & SCREEN MANAGER
# =============================================================
main_window = None
current_screen = None

def initialize_main_window():
    global main_window
    main_window = tk.Tk()
    main_window.title("ZUDO – AI WORD GAME")
    style_window(main_window)
    return main_window

def show_screen(screen_name, *args):
    global current_screen
    if current_screen:
        current_screen.pack_forget()

    if screen_name == "start":
        current_screen = create_start_screen(main_window)
    elif screen_name == "player_entry":
        current_screen = create_player_entry_screen(main_window)
    elif screen_name == "avatar_picker":
        current_screen = create_avatar_picker_screen(main_window, args[0])
    elif screen_name == "round_intro":
        current_screen = create_round_intro_screen(main_window, args[0])
    elif screen_name == "game":
        current_screen = create_game_screen(main_window, args[0])
    elif screen_name == "round_end":
        current_screen = create_round_end_screen(main_window, args[0], args[1])
    elif screen_name == "results":
        current_screen = create_results_screen(main_window)

    if current_screen:
        current_screen.pack(fill="both", expand=True)
    main_window.update_idletasks()

# =============================================================
#  SHARED HELPERS
# =============================================================

def style_window(win):
    win.configure(bg=BG_DEEP)
    cx = (win.winfo_screenwidth()  - WIN_W) // 2
    cy = (win.winfo_screenheight() - WIN_H) // 2
    win.geometry(f"{WIN_W}x{WIN_H}+{cx}+{cy}")
    win.resizable(False, False)

def make_card(parent, vivid=True, **kw):
    border = BORDER_VIVID if vivid else BORDER_DIM
    return tk.Frame(parent, bg=BG_CARD,
                    highlightbackground=border,
                    highlightthickness=1, **kw)

def neon_btn(parent, text, color, cmd, width=14, pady=8):
    b = tk.Button(parent, text=text, fg=BG_DEEP, bg=color,
                  font=F_BTN, bd=0, relief="flat",
                  activebackground=WHITE, activeforeground=BG_DEEP,
                  cursor="hand2", width=width, pady=pady, command=cmd)
    b.bind("<Enter>", lambda e: b.config(bg=WHITE))
    b.bind("<Leave>", lambda e: b.config(bg=color))
    return b

def player_color(name):
    idx = players.index(name) if name in players else 0
    return PLAYER_COLORS[idx % len(PLAYER_COLORS)]

FLOAT_CHARS = list("ABCDEFGHIJKLMNOPQRSTUVWXYZ0123456789अआइईउऊऋॠऌॡएऐओऔqwertyuioplkjhhfdsazxcvbnmyc.")

def spawn_letters(canvas, count=50, w=None, h=None):
    W = w or WIN_W
    H = h or WIN_H
    items = []
    for _ in range(count):
        x   = random.randint(0, W)
        y   = random.randint(0, H)
        ch  = random.choice(FLOAT_CHARS)
        col = random.choice([PURPLE, CYAN, BORDER_VIVID, "#3b3a7a", "#1e6b7a"])
        sz  = random.choice([10, 15, 20])
        s   = canvas.create_text(x, y, text=ch, fill=col,
                                 font=("Courier New", sz, "bold"))
        items.append(s)
    return items

def animate_letters(canvas, items, parent_widget, W=None, H=None):
    W = W or WIN_W
    H = H or WIN_H
    for s in items:
        canvas.move(s, 0, 1)
        coords = canvas.coords(s)
        if coords and coords[1] > H + 10:
            canvas.coords(s, random.randint(0, W), -10)
            canvas.itemconfig(s,
                text=random.choice(FLOAT_CHARS),
                fill=random.choice([PURPLE, CYAN, BORDER_VIVID, "#3b3a7a", "#1e6b7a"]))
    parent_widget.after(60, animate_letters, canvas, items, parent_widget, W, H)

def draw_scanlines(canvas, W=None, H=None):
    W = W or WIN_W
    H = H or WIN_H
    for y in range(0, H, 4):
        canvas.create_line(0, y, W, y, fill="#ffffff", stipple="gray12")

def base_canvas(frame, W=None, H=None):
    W = W or WIN_W
    H = H or WIN_H
    cv = tk.Canvas(frame, width=W, height=H,
                   bg=BG_DEEP, highlightthickness=0)
    cv.pack(fill="both", expand=True)
    draw_scanlines(cv, W, H)
    lts = spawn_letters(cv, count=55, w=W, h=H)
    animate_letters(cv, lts, frame, W, H)
    return cv

# ── FIXED: Avatar image loader — no subsample if image is small ──
def load_avatar_image(gender, size=160):
    path = BOY_AVATAR_PATH if gender == "boy" else GIRL_AVATAR_PATH
    try:
        img = tk.PhotoImage(file=path)
        w, h = img.width(), img.height()
        # Only subsample if image is larger than target size
        factor = max(1, min(w, h) // size)
        if factor > 1:
            return img.subsample(factor, factor)
        return img
    except Exception:
        return None

def load_avatar_image_zoom(gender, target_size=160):
    """Load avatar and zoom up if smaller than target, subsample if larger."""
    path = BOY_AVATAR_PATH if gender == "boy" else GIRL_AVATAR_PATH
    try:
        img = tk.PhotoImage(file=path)
        w, h = img.width(), img.height()
        smallest = min(w, h)
        if smallest == 0:
            return None
        if smallest < target_size:
            # zoom up
            factor = max(1, target_size // smallest)
            return img.zoom(factor, factor)
        else:
            factor = max(1, smallest // target_size)
            return img.subsample(factor, factor)
    except Exception:
        return None

def load_coin_image(size=120):
    try:
        img    = tk.PhotoImage(file=COIN_IMAGE_PATH)
        factor = max(1, img.width() // size)
        return img.subsample(factor, factor) if factor > 1 else img
    except Exception:
        return None


# =============================================================
#  SCREEN 1 – START / WELCOME
# =============================================================
def create_start_screen(parent):
    screen = tk.Frame(parent, bg=BG_DEEP)
    cv = base_canvas(screen)

    for (x1,y1,x2,y2) in [(0,0,120,3),(0,0,3,120),(WIN_W-120,0,WIN_W,3),(WIN_W-3,0,WIN_W,120),
                           (0,WIN_H-3,120,WIN_H),(0,WIN_H-120,3,WIN_H),
                           (WIN_W-120,WIN_H-3,WIN_W,WIN_H),(WIN_W-3,WIN_H-120,WIN_W,WIN_H)]:
        cv.create_rectangle(x1,y1,x2,y2, fill=CYAN, outline="")

    cv.create_rectangle(220, 110, 880, 260, outline=BORDER_VIVID, width=2, fill=BG_PANEL)
    cv.create_rectangle(224, 114, 876, 256, outline=PINK, width=1, fill="")
    cv.create_rectangle(226, 116, 874, 180, fill="#0f0d35", outline="")

    lbl = tk.Label(screen, text="Z U D O", font=("Courier New", 52, "bold"), fg=PINK, bg=BG_PANEL)
    cv.create_window(WIN_W//2, 168, window=lbl)
    cv.create_text(WIN_W//2, 228, text="A  I     W  O  R  D     G  A  M  E",
                   fill=CYAN, font=("Courier New", 14, "bold"))

    blink_texts = ["⚡  WELCOME  !", "🎯  NAMASTE  !", "🔥  NAMASKAR  !"]
    bi  = [0]
    sub = tk.Label(screen, text=blink_texts[0], font=("Courier New", 15, "bold"), fg=YELLOW, bg=BG_DEEP)
    cv.create_window(WIN_W//2, 310, window=sub)
    def blink():
        bi[0] = (bi[0]+1) % len(blink_texts)
        sub.config(text=blink_texts[bi[0]])
        screen.after(800, blink)
    blink()

    btn = neon_btn(screen, "▶   START  GAME", YELLOW,
                   cmd=lambda: show_screen("player_entry"), width=22)
    cv.create_window(WIN_W//2, 420, window=btn, height=50)

    cv.create_text(WIN_W//2, WIN_H - 18, text="© ZUDO AI  •  2026  •  Type. Think. Win.",
                   fill=BORDER_VIVID, font=F_SMALL)

    return screen


# =============================================================
#  SCREEN 2 – PLAYER NAME ENTRY
# =============================================================
def create_player_entry_screen(parent):
    global players, player_avatars, scores

    screen = tk.Frame(parent, bg=BG_DEEP)
    cv = base_canvas(screen)

    cv.create_text(WIN_W//2, 40, text="A D D   P L A Y E R S",
                   fill=PINK, font=("Courier New", 20, "bold"))
    cv.create_text(WIN_W//2, 68,
                   text=f"Enter names for 2 to {MAX_PLAYERS} players  •  Each player picks an avatar next",
                   fill=PURPLE, font=F_SMALL)

    outer = tk.Frame(parent, bg=BG_DEEP)
    cv.create_window(WIN_W//2, 350, window=outer, width=740, height=460)

    colors = [PINK, CYAN, GREEN, PURPLE, YELLOW, ORANGE]
    icons  = ["🎮","🕹","🎯","⭐","🔥","💎"]

    name_vars = ["Player 1", "Player 2"]
    live_entries = []

    def save_current_values():
        for i, e in enumerate(live_entries):
            if i < len(name_vars):
                name_vars[i] = e.get()

    def rebuild_rows():
        live_entries.clear()
        for widget in outer.winfo_children():
            widget.destroy()

        for idx, name in enumerate(name_vars):
            color = colors[idx % len(colors)]

            row = tk.Frame(outer, bg=BG_CARD,
                           highlightbackground=color, highlightthickness=1)
            row.pack(fill="x", pady=5, padx=4)

            tk.Label(row, text=f"P{idx+1}", fg=color, bg=BG_CARD,
                     font=("Courier New", 14, "bold"), width=3).pack(side="left", padx=(10,0))
            tk.Label(row, text=icons[idx % len(icons)],
                     fg=color, bg=BG_CARD,
                     font=("Courier New", 14)).pack(side="left", padx=(4, 8))

            can_remove = (len(name_vars) > 2)

            def make_remove(i):
                def do_remove():
                    save_current_values()
                    if len(name_vars) <= 2:
                        flash_msg("⚠  Minimum 2 players required!")
                        return
                    name_vars.pop(i)
                    rebuild_rows()
                return do_remove

            rm_btn = tk.Button(row, text="✕  REMOVE",
                               fg=RED if can_remove else BORDER_DIM,
                               bg=BG_CARD,
                               font=("Courier New", 10, "bold"),
                               bd=0, relief="flat",
                               cursor="hand2" if can_remove else "",
                               activebackground=BG_CARD, activeforeground=WHITE,
                               padx=10, pady=6,
                               command=make_remove(idx) if can_remove else lambda: None)
            rm_btn.pack(side="right", padx=(0, 10))

            e = tk.Entry(row, font=("Courier New", 15, "bold"),
                         bg=BG_DEEP, fg=WHITE, bd=0,
                         insertbackground=color,
                         highlightbackground=color,
                         highlightthickness=1,
                         width=24)
            e.insert(0, name)
            e.bind("<FocusIn>", lambda ev, ew=e: ew.select_range(0, tk.END))
            e.pack(side="left", ipady=9, padx=(0, 6), fill="x", expand=True)
            live_entries.append(e)

        if live_entries:
            live_entries[-1].focus()

    flash_lbl = tk.Label(screen, text="", fg=RED, bg=BG_DEEP,
                         font=("Courier New", 10, "bold"))
    cv.create_window(WIN_W//2, WIN_H - 76, window=flash_lbl)

    def flash_msg(msg, color=RED):
        flash_lbl.config(text=msg, fg=color)
        screen.after(2400, lambda: flash_lbl.config(text=""))

    def add_player():
        save_current_values()
        if len(name_vars) >= MAX_PLAYERS:
            flash_msg(f"⚠  Maximum {MAX_PLAYERS} players allowed!")
            return
        name_vars.append(f"Player {len(name_vars)+1}")
        rebuild_rows()

    def confirm_names():
        global players, player_avatars, scores
        save_current_values()
        names = [n.strip() for n in name_vars if n.strip()]
        if len(names) < 2:
            flash_msg("⚠  Enter at least 2 player names!")
            return
        seen = set(); unique = []
        for n in names:
            if n.lower() not in seen:
                seen.add(n.lower()); unique.append(n)
        players        = unique
        player_avatars = {}
        scores         = {p: 0 for p in players}
        show_screen("avatar_picker", 0)

    rebuild_rows()

    btn_row = tk.Frame(screen, bg=BG_DEEP)
    cv.create_window(WIN_W//2, WIN_H - 38, window=btn_row)

    neon_btn(btn_row, "+ ADD PLAYER", CYAN,   add_player,    width=14).pack(side="left", padx=8)
    neon_btn(btn_row, "▶  NEXT",      YELLOW, confirm_names, width=12).pack(side="left", padx=8)

    return screen


# =============================================================
#  SCREEN 2b – AVATAR PICKER  (FULLY REWORKED GRAPHICS)
# =============================================================
def create_avatar_picker_screen(parent, player_idx):
    player_name = players[player_idx]
    p_color     = PLAYER_COLORS[player_idx % len(PLAYER_COLORS)]

    screen = tk.Frame(parent, bg=BG_DEEP)
    cv = base_canvas(screen)

    # Title
    cv.create_text(WIN_W//2, 38,
                   text=f"{player_name.upper()}   —   CHOOSE  YOUR  AVATAR",
                   fill=p_color, font=("Courier New", 18, "bold"))
    cv.create_text(WIN_W//2, 64, text=f"Player {player_idx+1} of {len(players)}",
                   fill=PURPLE, font=F_SMALL)

    # Progress dots
    dot_gap = 28
    dot_start = WIN_W//2 - (len(players) * dot_gap)//2
    for di in range(len(players)):
        fill = p_color if di <= player_idx else BORDER_DIM
        cv.create_oval(dot_start + di*dot_gap, 78,
                       dot_start + di*dot_gap + 16, 94,
                       fill=fill, outline="")

    # Load images using the improved loader
    boy_img  = load_avatar_image_zoom("boy",  target_size=150)
    girl_img = load_avatar_image_zoom("girl", target_size=150)

    # Keep references so garbage collector doesn't kill them
    cv._boy_img  = boy_img
    cv._girl_img = girl_img

    selected = [None]

    # Layout: two avatar circles side by side, centred
    CIRCLE_R   = 105          # radius of the outer ring
    IMG_R      = 88           # radius of the inner avatar area
    cx_boy     = WIN_W//2 - 200
    cx_girl    = WIN_W//2 + 200
    circle_cy  = 310

    def draw_card(gender):
        is_sel    = (selected[0] == gender)
        img       = boy_img  if gender == "boy"  else girl_img
        label     = " "  if gender == "boy"  else " "
        label_col = CYAN     if gender == "boy"  else PINK
        cx_       = cx_boy   if gender == "boy"  else cx_girl
        tag       = f"card_{gender}"
        ring_col  = p_color  if is_sel else BORDER_DIM

        # ── Outer ambient glow rings (selected only) ────────
        if is_sel:
            for off, dash in [(CIRCLE_R+26, (2,8)), (CIRCLE_R+14, (3,6))]:
                cv.create_oval(cx_-off, circle_cy-off, cx_+off, circle_cy+off,
                               outline=ring_col, width=1, fill="", dash=dash, tags=(tag,))

        # ── Main ring ───────────────────────────────────────
        # Outer ring border
        cv.create_oval(cx_-CIRCLE_R, circle_cy-CIRCLE_R,
                       cx_+CIRCLE_R, circle_cy+CIRCLE_R,
                       outline=ring_col,
                       width=3 if is_sel else 1,
                       fill=BG_PANEL if is_sel else BG_DEEP,
                       tags=(tag,))

        # ── Inner avatar circle — clean, NO rectangle box ───
        # Background fill for avatar area
        cv.create_oval(cx_-IMG_R, circle_cy-IMG_R,
                       cx_+IMG_R, circle_cy+IMG_R,
                       fill=BG_CARD,
                       outline=ring_col if is_sel else "#1a1850",
                       width=1,
                       tags=(tag,))

        # Avatar image rendered inside the circle (no clipping in tkinter,
        # but the circle visually frames it cleanly)
        if img:
            cv.create_image(cx_, circle_cy, image=img, anchor="center", tags=(tag,))
        else:
            # Fallback symbol
            fb   = "♂" if gender == "boy" else "♀"
            fcol = CYAN if gender == "boy" else PINK
            cv.create_text(cx_, circle_cy, text=fb, fill=fcol,
                           font=("Courier New", 72, "bold"), tags=(tag,))

        # ── Gender label below circle ────────────────────────
        label_y = circle_cy + CIRCLE_R + 28
        cv.create_text(cx_, label_y, text=label,
                       fill=p_color if is_sel else label_col,
                       font=("Courier New", 17, "bold"), tags=(tag,))

        # ── SELECTED pill badge ──────────────────────────────
        if is_sel:
            badge_y = label_y + 26
            cv.create_rectangle(cx_-64, badge_y-11,
                                 cx_+64, badge_y+11,
                                 fill=GREEN, outline="", tags=(tag,))
            cv.create_text(cx_, badge_y, text="✔  SELECTED",
                           fill=BG_DEEP, font=("Courier New", 10, "bold"), tags=(tag,))

        # ── Invisible hit area ───────────────────────────────
        hit = cv.create_rectangle(cx_-CIRCLE_R-20, circle_cy-CIRCLE_R-20,
                                   cx_+CIRCLE_R+20, circle_cy+CIRCLE_R+60,
                                   fill="", outline="", tags=(tag, "hit"))
        cv.tag_bind(hit, "<Button-1>", lambda e, g=gender: on_select(g))
        cv.tag_bind(hit, "<Enter>",    lambda e: cv.config(cursor="hand2"))
        cv.tag_bind(hit, "<Leave>",    lambda e: cv.config(cursor=""))

    def redraw():
        cv.delete("card_boy")
        cv.delete("card_girl")
        draw_card("boy")
        draw_card("girl")

    def on_select(gender):
        play_sound(click_sound)
        selected[0] = gender
        redraw()
        confirm_btn.config(state="normal", bg=GREEN)
        confirm_btn.bind("<Enter>", lambda e: confirm_btn.config(bg=WHITE))
        confirm_btn.bind("<Leave>", lambda e: confirm_btn.config(bg=GREEN))

    redraw()

    # ── Confirm button ────────────────────────────────────────
    confirm_btn = tk.Button(screen, text="▶  CONFIRM  AVATAR",
                            fg=BG_DEEP, bg=BORDER_DIM,
                            font=F_BTN, bd=0, relief="flat",
                            cursor="hand2", width=24, pady=10, state="disabled")
    cv.create_window(WIN_W//2, WIN_H - 38, window=confirm_btn, height=46)

    def confirm():
        if selected[0] is None:
            messagebox.showwarning("No Choice", "Please select Boy or Girl!")
            return
        play_sound(click_sound)
        player_avatars[player_name] = selected[0]
        next_idx = player_idx + 1
        if next_idx < len(players):
            show_screen("avatar_picker", next_idx)
        else:
            show_screen("round_intro", 1)

    confirm_btn.config(command=confirm)
    return screen


# =============================================================
#  SCREEN 3 – ROUND INTRO
# =============================================================
def create_round_intro_screen(parent, round_num):
    screen = tk.Frame(parent, bg=BG_DEEP)
    cv = base_canvas(screen)

    particles = []
    for _ in range(80):
        angle = random.uniform(0, 2 * math.pi)
        speed = random.uniform(1.5, 5.0)
        col   = random.choice([PINK, CYAN, YELLOW, GREEN, PURPLE, ORANGE])
        sz    = random.randint(2, 5)
        p = {
            "x": WIN_W // 2, "y": WIN_H // 2,
            "vx": math.cos(angle) * speed, "vy": math.sin(angle) * speed,
            "col": col, "sz": sz, "life": 1.0,
            "id": cv.create_oval(WIN_W//2, WIN_H//2, WIN_W//2+sz, WIN_H//2+sz, fill=col, outline="")
        }
        particles.append(p)

    def animate_particles():
        alive = []
        for p in particles:
            p["x"] += p["vx"]; p["y"] += p["vy"]
            p["vy"] += 0.08;   p["life"] -= 0.012
            if p["life"] > 0:
                s = p["sz"]
                cv.coords(p["id"], p["x"]-s*p["life"], p["y"]-s*p["life"],
                           p["x"]+s*p["life"], p["y"]+s*p["life"])
                alive.append(p)
            else:
                cv.delete(p["id"])
        particles[:] = alive
        if alive: screen.after(16, animate_particles)
    screen.after(200, animate_particles)

    ring_angle = [0]; ring_ids = []
    for ri in range(3):
        rid = cv.create_arc(WIN_W//2-180, WIN_H//2-180, WIN_W//2+180, WIN_H//2+180,
                             start=ri*120, extent=80,
                             outline=[CYAN, PINK, YELLOW][ri], width=3+ri, style="arc")
        ring_ids.append(rid)

    def spin_ring():
        ring_angle[0] = (ring_angle[0] + 3) % 360
        for ri, rid in enumerate(ring_ids):
            cv.itemconfig(rid, start=ring_angle[0] + ri*120)
        screen.after(20, spin_ring)
    spin_ring()

    cx, cy = WIN_W//2, WIN_H//2
    cv.create_oval(cx-140, cy-140, cx+140, cy+140, fill=BG_PANEL, outline=BORDER_VIVID, width=2)
    cv.create_text(cx, cy - 48, text="R O U N D", fill=PURPLE, font=("Courier New", 14, "bold"))
    cv.create_text(cx, cy + 10, text=str(round_num), fill=YELLOW, font=("Courier New", 72, "bold"))
    cv.create_text(cx, cy + 66, text=f"of  {ROUND_LIMIT}", fill=CYAN, font=("Courier New", 13, "bold"))

    lts = spawn_letters(cv, count=30)
    animate_letters(cv, lts, screen)

    slogans = ["Think fast. Type faster.", "Words are your weapon.",
               "Only the sharp survive.", "Brain cells activated!"]
    slogan_lbl = tk.Label(screen, text=random.choice(slogans), fg=PINK, bg=BG_DEEP,
                          font=("Courier New", 12, "bold"))
    cv.create_window(WIN_W//2, WIN_H//2 + 160, window=slogan_lbl)

    if round_num > 1:
        sf = tk.Frame(screen, bg=BG_PANEL, highlightbackground=BORDER_VIVID, highlightthickness=1)
        cv.create_window(WIN_W - 150, WIN_H//2 - 20, window=sf, width=250, height=260)
        tk.Label(sf, text="SCORES  SO  FAR", fg=YELLOW, bg=BG_PANEL,
                 font=("Courier New", 9, "bold")).pack(pady=(8, 4))
        tk.Frame(sf, bg=BORDER_VIVID, height=1).pack(fill="x", padx=10)
        sorted_p = sorted(players, key=lambda x: scores[x], reverse=True)
        medals   = ["🥇","🥈","🥉"]
        for rank, p in enumerate(sorted_p):
            col = PLAYER_COLORS[players.index(p) % len(PLAYER_COLORS)]
            r   = tk.Frame(sf, bg=BG_CARD, highlightbackground=col, highlightthickness=1)
            r.pack(fill="x", pady=3, padx=8)
            tk.Label(r, text=medals[rank] if rank < 3 else f"#{rank+1}",
                     fg=YELLOW, bg=BG_CARD, font=("Courier New", 9), width=3).pack(side="left")
            tk.Label(r, text=p[:10].upper(), fg=col, bg=BG_CARD,
                     font=("Courier New", 9, "bold")).pack(side="left", padx=4)
            tk.Label(r, text=f"{scores[p]} pts", fg=WHITE, bg=BG_CARD,
                     font=("Courier New", 9)).pack(side="right", padx=6)

    countdown_lbl = tk.Label(screen, text="", fg=PINK, bg=BG_DEEP,
                             font=("Courier New", 13, "bold"))
    cv.create_window(WIN_W//2, WIN_H - 88, window=countdown_lbl)

    ready_texts = ["⚡  GET  READY  !", "🎯  PREPARE  !", "🔥  HERE  WE  GO  !"]
    ri_b = [0]
    def blink_ready():
        ri_b[0] = (ri_b[0]+1) % len(ready_texts)
        countdown_lbl.config(text=ready_texts[ri_b[0]])
        screen.after(650, blink_ready)
    blink_ready()

    def go():
        play_sound(click_sound)
        show_screen("game", round_num)

    btn = neon_btn(screen, f"▶   BEGIN  ROUND  {round_num}", GREEN, go, width=28)
    cv.create_window(WIN_W//2, WIN_H - 44, window=btn, height=50)

    return screen


# =============================================================
#  TURN SPLASH
# =============================================================
def show_turn_splash_on_canvas(parent_frame, player_name, on_done):
    p_col  = player_color(player_name)
    gender = player_avatars.get(player_name, "boy")

    ov = tk.Canvas(parent_frame, width=WIN_W, height=WIN_H,
                   bg=BG_DEEP, highlightthickness=0)
    ov.place(x=0, y=0, width=WIN_W, height=WIN_H)
    ov.create_rectangle(0, 0, WIN_W, WIN_H, fill=BG_PANEL, stipple="gray50")

    # Halo rings around avatar
    for off in range(80, 0, -10):
        ov.create_oval(WIN_W//2 - off, WIN_H//2 - 180 - off,
                        WIN_W//2 + off, WIN_H//2 - 20 + off,
                        outline=p_col if off == 80 else BORDER_DIM, width=1, fill="")

    cx, cy = WIN_W//2, WIN_H//2 - 60

    # Load avatar with improved loader
    av_img = load_avatar_image_zoom(gender, target_size=150)
    if av_img:
        ov.create_image(cx, cy - 10, image=av_img, anchor="center")
        ov._av_img = av_img
    else:
        fb = "♂" if gender == "boy" else "♀"
        ov.create_text(cx, cy - 10, text=fb, fill=p_col,
                       font=("Courier New", 90, "bold"))

    ov.create_text(cx, cy + 120, text=player_name.upper(),
                   fill=p_col, font=("Courier New", 28, "bold"))
    ov.create_text(cx, cy + 158, text="—  Y O U R   T U R N  —",
                   fill=YELLOW, font=("Courier New", 12, "bold"))

    count_lbl_id = ov.create_text(cx, cy + 210, text="Get ready…",
                                   fill=WHITE, font=("Courier New", 22, "bold"))
    counts = ["3", "2", "1", "GO !"]
    ci = [0]

    def tick():
        if ci[0] < len(counts):
            col = GREEN if counts[ci[0]] == "GO !" else WHITE
            ov.itemconfig(count_lbl_id, text=counts[ci[0]], fill=col)
            ci[0] += 1
            parent_frame.after(700, tick)
        else:
            ov.destroy()
            on_done()

    parent_frame.after(400, tick)


# =============================================================
#  SCREEN 4 – MAIN GAME WINDOW
# =============================================================
def create_game_screen(parent, current_round=1):
    screen = tk.Frame(parent, bg=BG_DEEP)

    data           = load_round_data(current_round)
    prompts        = list(data["category"].unique()) if not data.empty else ["General"]
    round_category = random.choice(prompts)

    state = {
        "player_idx"    : 0,
        "time_left"     : TIME_LIMIT,
        "timer_id"      : None,
        "timer_running" : False,
        "used_words"    : set(),
        "splash_active" : False,
        "warning_played": False,
    }

    # ── TOP BAR ──────────────────────────────────────────────
    top = tk.Frame(screen, bg=BG_DEEP)
    top.pack(fill="x", padx=24, pady=(14, 0))

    tk.Label(top, text="ZUDO", fg=PINK, bg=BG_DEEP,
             font=("Courier New", 22, "bold")).pack(side="left")
    tk.Label(top, text="  AI WORD GAME", fg=CYAN, bg=BG_DEEP,
             font=("Courier New", 10, "bold")).pack(side="left")
    tk.Label(top, text=f"ROUND  {current_round}  /  {ROUND_LIMIT}",
             fg=YELLOW, bg=BG_DEEP,
             font=("Courier New", 12, "bold")).pack(side="right")

    tk.Frame(screen, bg=BORDER_VIVID, height=2).pack(fill="x", padx=24, pady=(8, 10))

    # ── BODY ──────────────────────────────────────────────────
    body = tk.Frame(screen, bg=BG_DEEP)
    body.pack(fill="both", expand=True, padx=24, pady=(0, 12))

    left = tk.Frame(body, bg=BG_DEEP)
    left.pack(side="left", fill="both", expand=True, padx=(0, 16))

    right = tk.Frame(body, bg=BG_DEEP, width=280)
    right.pack(side="right", fill="y")
    right.pack_propagate(False)

    # ── LEFT: Player banner ───────────────────────────────────
    player_banner = tk.Frame(left, bg=BG_CARD,
                             highlightbackground=BORDER_VIVID, highlightthickness=2)
    player_banner.pack(fill="x", pady=(0, 10))

    accent_bar = tk.Frame(player_banner, bg=PINK, width=6)
    accent_bar.pack(side="left", fill="y")

    banner_inner = tk.Frame(player_banner, bg=BG_CARD, padx=18, pady=14)
    banner_inner.pack(side="left", fill="both", expand=True)

    tk.Label(banner_inner, text="NOW  PLAYING", fg=PURPLE, bg=BG_CARD,
             font=("Courier New", 9, "bold")).pack(anchor="w")

    lbl_name = tk.Label(banner_inner, text="", fg=PINK, bg=BG_CARD,
                        font=("Courier New", 30, "bold"))
    lbl_name.pack(anchor="w")

    lbl_score_inline = tk.Label(banner_inner, text="", fg=YELLOW, bg=BG_CARD,
                                font=("Courier New", 11))
    lbl_score_inline.pack(anchor="w", pady=(2, 8))

    cat_frame = tk.Frame(banner_inner, bg="#0a0830",
                         highlightbackground=YELLOW, highlightthickness=1)
    cat_frame.pack(anchor="w")
    tk.Label(cat_frame, text="  CATEGORY: ", fg=PURPLE, bg="#0a0830",
             font=("Courier New", 10)).pack(side="left")
    tk.Label(cat_frame, text=f" {round_category.upper()} ",
             fg=YELLOW, bg="#0a0830",
             font=("Courier New", 14, "bold")).pack(side="left")

    def update_avatar(name):
        col = player_color(name)
        accent_bar.config(bg=col)
        player_banner.config(highlightbackground=col)

    # ── LEFT: Timer ────────────────────────────────────────────
    timer_card = make_card(left, padx=18, pady=12)
    timer_card.pack(fill="x", pady=(0, 10))

    t_row = tk.Frame(timer_card, bg=BG_CARD)
    t_row.pack(fill="x")

    tk.Label(t_row, text="⏱  TIME LEFT", fg=PURPLE, bg=BG_CARD,
             font=("Courier New", 10, "bold")).pack(side="left", anchor="s", pady=(0, 4))

    lbl_timer = tk.Label(t_row, text=str(TIME_LIMIT), fg=GREEN, bg=BG_CARD, font=F_TIMER)
    lbl_timer.pack(side="left", padx=(14, 4))

    tk.Label(t_row, text="SEC", fg=PURPLE, bg=BG_CARD,
             font=("Courier New", 11)).pack(side="left", anchor="s", pady=(0, 10))

    bar_bg   = tk.Frame(timer_card, bg=BORDER_DIM, height=10)
    bar_bg.pack(fill="x", pady=(8, 0))
    bar_fill = tk.Frame(bar_bg, bg=GREEN, height=10)
    bar_fill.place(x=0, y=0, relwidth=1.0, height=10)

    def update_bar(frac, col):
        bar_fill.config(bg=col)
        bar_fill.place(x=0, y=0, relwidth=max(0.0, frac), height=10)

    # ── LEFT: Warning flash label ──────────────────────────────
    warn_lbl = tk.Label(left, text="", fg=RED, bg=BG_DEEP,
                        font=("Courier New", 15, "bold"))
    warn_lbl.pack(fill="x", pady=(0, 4))
    warn_blink_id = [None]

    def flash_warning(msg, color=RED, blinks=6):
        if warn_blink_id[0]: screen.after_cancel(warn_blink_id[0])
        count = [0]
        def blink_it():
            count[0] += 1
            if count[0] > blinks * 2:
                warn_lbl.config(text="")
                return
            warn_lbl.config(text=msg if count[0] % 2 == 1 else "", fg=color)
            warn_blink_id[0] = screen.after(200, blink_it)
        blink_it()

    # ── LEFT: Word input ───────────────────────────────────────
    inp_card = make_card(left, padx=18, pady=14)
    inp_card.pack(fill="x", pady=(0, 8))

    tk.Label(inp_card, text="TYPE  YOUR  WORD", fg=CYAN, bg=BG_CARD,
             font=("Courier New", 11, "bold")).pack(anchor="w")

    word_entry = tk.Entry(inp_card, font=("Courier New", 22, "bold"),
                          bg=BG_DEEP, fg=WHITE, bd=0,
                          insertbackground=CYAN,
                          highlightbackground=BORDER_VIVID, highlightthickness=2)
    word_entry.pack(fill="x", ipady=12, pady=(6, 10))

    br = tk.Frame(inp_card, bg=BG_CARD)
    br.pack(fill="x")
    neon_btn(br, "▶  SUBMIT", PINK,   lambda: submit_word(), width=16, pady=10
             ).pack(side="left", padx=(0, 12))
    neon_btn(br, "SKIP  ▷",  PURPLE,  lambda: skip_turn(),   width=12, pady=10
             ).pack(side="left")
    word_entry.bind("<Return>", lambda e: submit_word())

    # ── RIGHT: Used words ─────────────────────────────────────
    uw_header = tk.Label(right, text="🔤  USED WORDS",
                         fg=PURPLE, bg=BG_DEEP,
                         font=("Courier New", 12, "bold"))
    uw_header.pack(anchor="w", pady=(0, 8))

    uw_count_lbl = tk.Label(right, text="0 words used",
                             fg=BORDER_VIVID, bg=BG_DEEP,
                             font=("Courier New", 9))
    uw_count_lbl.pack(anchor="w", pady=(0, 6))

    uw_card = make_card(right, vivid=False, padx=10, pady=8)
    uw_card.pack(fill="both", expand=True)

    uw_canvas = tk.Canvas(uw_card, bg=BG_CARD, highlightthickness=0)
    uw_sb = tk.Scrollbar(uw_card, orient="vertical", command=uw_canvas.yview)
    uw_inner = tk.Frame(uw_canvas, bg=BG_CARD)
    uw_inner.bind("<Configure>",
                  lambda e: uw_canvas.configure(scrollregion=uw_canvas.bbox("all")))
    uw_canvas.create_window((0, 0), window=uw_inner, anchor="nw")
    uw_canvas.configure(yscrollcommand=uw_sb.set)
    uw_canvas.pack(side="left", fill="both", expand=True)
    uw_sb.pack(side="right", fill="y")

    def refresh_used():
        for w in uw_inner.winfo_children():
            w.destroy()
        uw_count_lbl.config(text=f"{len(state['used_words'])} word{'s' if len(state['used_words']) != 1 else ''} used")
        row_f = [None]; ci = [0]
        def nr():
            row_f[0] = tk.Frame(uw_inner, bg=BG_CARD)
            row_f[0].pack(anchor="w")
        nr()
        for word in sorted(state["used_words"]):
            if ci[0] >= 2:
                nr(); ci[0] = 0
            tk.Label(row_f[0], text=word, fg=CYAN, bg="#0f2a3a",
                     font=("Courier New", 8, "bold"),
                     padx=5, pady=3,
                     highlightbackground=CYAN,
                     highlightthickness=1).pack(side="left", padx=2, pady=2)
            ci[0] += 1

    # ── Correct word popup ─────────────────────────────────────
    def show_correct_popup(points, word):
        play_sound(coin_sound)

        ov = tk.Canvas(screen, width=WIN_W, height=WIN_H,
                       bg=BG_DEEP, highlightthickness=0)
        ov.place(x=0, y=0, width=WIN_W, height=WIN_H)
        ov.create_rectangle(0, 0, WIN_W, WIN_H, fill=BG_PANEL, stipple="gray50")

        cx, cy = WIN_W // 2, WIN_H // 2

        sparks = []
        for _ in range(28):
            angle = random.uniform(0, 2 * math.pi)
            speed = random.uniform(2.5, 7.0)
            col   = random.choice([YELLOW, GOLD, ORANGE, WHITE])
            sid   = ov.create_text(cx, cy, text="★",
                                   fill=col, font=("Courier New", random.randint(8,16), "bold"))
            sparks.append({"x": cx, "y": cy,
                           "vx": math.cos(angle)*speed, "vy": math.sin(angle)*speed - 1.5,
                           "life": 1.0, "id": sid})

        def animate_sparks():
            alive = []
            for s in sparks:
                s["x"] += s["vx"]; s["y"] += s["vy"]
                s["vy"] += 0.2;    s["life"] -= 0.03
                if s["life"] > 0:
                    ov.coords(s["id"], s["x"], s["y"])
                    alive.append(s)
                else:
                    ov.delete(s["id"])
            sparks[:] = alive
            if alive: screen.after(16, animate_sparks)
        animate_sparks()

        for off in [70, 52, 34]:
            ov.create_rectangle(cx - 230 - off//4, cy - 115 - off//4,
                                 cx + 230 + off//4, cy + 115 + off//4,
                                 outline=YELLOW if off == 70 else GOLD,
                                 width=1, fill="", dash=(3, 7))

        ov.create_rectangle(cx - 230, cy - 115, cx + 230, cy + 115,
                             fill=BG_CARD, outline=YELLOW, width=3)

        coin_img = load_coin_image(size=110)
        ov._coin_img = coin_img
        if coin_img:
            ov.create_image(cx - 148, cy, image=coin_img, anchor="center")
        else:
            ov.create_oval(cx-190, cy-52, cx-106, cy+52, fill=GOLD, outline=YELLOW, width=3)
            ov.create_text(cx - 148, cy, text="$", fill=BG_DEEP,
                           font=("Courier New", 32, "bold"))

        ov.create_text(cx + 40, cy - 44, text="✅  CORRECT !",
                       fill=GREEN, font=("Courier New", 20, "bold"))
        ov.create_text(cx + 40, cy + 4, text=f'"{word.upper()}"',
                       fill=CYAN, font=("Courier New", 15, "bold"))
        ov.create_text(cx + 40, cy + 48, text=f"+ {points}  POINTS",
                       fill=YELLOW, font=("Courier New", 20, "bold"))

        screen.after(1200, ov.destroy)

    # ── game logic ────────────────────────────────────────────
    def update_ui():
        p   = players[state["player_idx"]]
        col = player_color(p)
        lbl_name.config(text=p.upper(), fg=col)
        lbl_score_inline.config(text=f"Score:  {scores[p]} pts")
        update_avatar(p)
        refresh_used()

    def run_timer():
        t   = state["time_left"]
        col = GREEN if t > 10 else (YELLOW if t > WARNING_AT else RED)
        lbl_timer.config(text=str(t), fg=col)
        update_bar(t / TIME_LIMIT, col)

        if t <= WARNING_AT:
            flash_col = RED if (t % 2 == 0) else BG_CARD
            timer_card.config(bg=flash_col)
            lbl_timer.config(bg=flash_col)

        if t == WARNING_AT and not state["warning_played"]:
            play_sound(warning_sound)
            state["warning_played"] = True
            flash_warning("⚠  LOW  TIME  —  HURRY  !", RED, blinks=10)

        if t <= 0:
            state["timer_running"] = False
            timer_card.config(bg=BG_CARD)
            lbl_timer.config(bg=BG_CARD)
            flash_warning(f"⏰  TIME UP !  {players[state['player_idx']].upper()}  —  NEXT PLAYER !", RED, blinks=4)
            screen.after(900, advance_turn)
            return

        state["time_left"]    -= 1
        state["timer_running"] = True
        state["timer_id"] = screen.after(1000, run_timer)

    def cancel_timer():
        if state["timer_id"]:
            screen.after_cancel(state["timer_id"])
            state["timer_id"] = None
        state["timer_running"] = False

    def start_turn():
        state["splash_active"]  = False
        state["warning_played"] = False
        timer_card.config(bg=BG_CARD)
        lbl_timer.config(bg=BG_CARD)
        warn_lbl.config(text="")
        word_entry.config(state="normal")
        word_entry.delete(0, tk.END)
        word_entry.focus()
        state["time_left"] = TIME_LIMIT
        update_ui()
        run_timer()

    def advance_turn():
        cancel_timer()
        timer_card.config(bg=BG_CARD)
        lbl_timer.config(bg=BG_CARD)
        word_entry.config(state="disabled")
        state["player_idx"] += 1
        if state["player_idx"] >= len(players):
            nxt = current_round + 1
            if nxt > ROUND_LIMIT:
                show_screen("results")
            else:
                show_screen("round_end", current_round, nxt)
            return
        state["splash_active"] = True
        next_p = players[state["player_idx"]]
        show_turn_splash_on_canvas(screen, next_p, on_done=start_turn)

    def submit_word():
        if not state["timer_running"] or state["splash_active"]:
            return
        player     = players[state["player_idx"]]
        word       = word_entry.get().strip()
        if not word:
            flash_warning("✏  TYPE  A  WORD  FIRST !", YELLOW, blinks=4)
            return
        word_lower = word.lower()

        if word_lower in {w.lower() for w in state["used_words"]}:
            play_sound(wrong_sound)
            flash_warning(f"❌  '{word.upper()}'  ALREADY  USED  —  TRY  ANOTHER !", RED, blinks=6)
            word_entry.delete(0, tk.END)
            word_entry.config(highlightbackground=RED)
            screen.after(800, lambda: word_entry.config(highlightbackground=BORDER_VIVID))
            return

        cat_words = data[data["category"] == round_category]["word"]
        valid = any(word_lower == w.lower() for w in cat_words)

        if valid:
            pts = max(1, state["time_left"])
            scores[player] += pts
            state["used_words"].add(word)
            cancel_timer()
            show_correct_popup(pts, word)
            word_entry.delete(0, tk.END)
            word_entry.config(state="disabled")
            screen.after(1150, advance_turn)
        else:
            play_sound(wrong_sound)
            flash_warning(f"❌  '{word.upper()}'  NOT  VALID  —  TRY  AGAIN !", RED, blinks=6)
            word_entry.delete(0, tk.END)
            word_entry.config(highlightbackground=RED)
            screen.after(600, lambda: word_entry.config(highlightbackground=BORDER_VIVID))

    def skip_turn():
        if not state["timer_running"] or state["splash_active"]:
            return
        cancel_timer()
        flash_warning("⏩  SKIPPED  —  NEXT  PLAYER !", PURPLE, blinks=3)
        screen.after(700, advance_turn)

    word_entry.config(state="disabled")
    show_turn_splash_on_canvas(screen, players[0], on_done=start_turn)

    return screen


# =============================================================
#  SCREEN 5 – ROUND END
# =============================================================
def create_round_end_screen(parent, finished_round, next_round):
    screen = tk.Frame(parent, bg=BG_DEEP)
    cv = base_canvas(screen)

    cv.create_rectangle(140, 42, 960, 150, outline=GREEN, width=2, fill=BG_PANEL)
    cv.create_rectangle(143, 45, 957, 147, outline=BORDER_VIVID, width=1, fill="")
    cv.create_text(WIN_W//2, 82, text=f"🏁  ROUND  {finished_round}  COMPLETE !",
                   fill=GREEN, font=("Courier New", 24, "bold"))
    cv.create_text(WIN_W//2, 128, text=f"Scores after {finished_round} of {ROUND_LIMIT} rounds  ▼",
                   fill=CYAN, font=("Courier New", 10))

    sorted_p = sorted(players, key=lambda x: scores[x], reverse=True)
    medals   = ["🥇", "🥈", "🥉"]

    sf = tk.Frame(screen, bg=BG_DEEP)
    cv.create_window(WIN_W//2, 380, window=sf, width=680, height=360)

    for rank, p in enumerate(sorted_p):
        col    = PLAYER_COLORS[players.index(p) % len(PLAYER_COLORS)]
        row    = tk.Frame(sf, bg=BG_CARD, highlightbackground=col, highlightthickness=1)
        row.pack(fill="x", pady=5)

        tk.Label(row, text=medals[rank] if rank < 3 else f"#{rank+1}",
                 fg=YELLOW, bg=BG_CARD,
                 font=("Courier New", 18), width=3).pack(side="left", padx=(14, 0), pady=14)
        tk.Label(row, text=p.upper(), fg=col, bg=BG_CARD,
                 font=("Courier New", 16, "bold")).pack(side="left", padx=14)
        tk.Label(row, text=f"{scores[p]} pts",
                 fg=YELLOW, bg=BG_CARD,
                 font=("Courier New", 16, "bold")).pack(side="right", padx=14)

    def go_next():
        play_sound(click_sound)
        show_screen("round_intro", next_round)

    btn = neon_btn(screen, f"▶   GO TO ROUND  {next_round}", CYAN, go_next, width=26)
    cv.create_window(WIN_W//2, WIN_H - 38, window=btn, height=48)

    return screen


# =============================================================
#  SCREEN 6 – RESULTS / GAME OVER
# =============================================================
def create_results_screen(parent):
    play_sound(win_sound)

    screen = tk.Frame(parent, bg=BG_DEEP)
    cv = base_canvas(screen)

    winner = max(scores, key=scores.get)
    w_col  = player_color(winner)
    gender = player_avatars.get(winner, "boy")

    cv.create_rectangle(80, 28, WIN_W-80, 120, outline=YELLOW, width=2, fill=BG_PANEL)
    cv.create_rectangle(83, 31, WIN_W-83, 117, outline=BORDER_VIVID, width=1, fill="")
    cv.create_text(WIN_W//2, 64, text="🎮   G A M E   O V E R   !",
                   fill=PINK, font=("Courier New", 24, "bold"))
    cv.create_text(WIN_W//2, 100, text=f"🏆   WINNER  :   {winner.upper()}   🏆",
                   fill=YELLOW, font=("Courier New", 14, "bold"))

    cx_, cy_ = WIN_W//2, 240
    arc_a  = [0]
    for ri in range(3):
        ai = cv.create_arc(cx_-80, cy_-80, cx_+80, cy_+80,
                            start=ri*120, extent=70,
                            outline=[w_col, CYAN, YELLOW][ri], width=4-ri, style="arc")
        def spin(arc_id=ai, offset=ri):
            arc_a[0] = (arc_a[0]+4) % 360
            cv.itemconfig(arc_id, start=arc_a[0]+offset*120)
            screen.after(20, spin)
        spin()

    cv.create_oval(cx_-56, cy_-56, cx_+56, cy_+56, fill=BG_CARD, outline=w_col, width=3)
    av_img = load_avatar_image_zoom(gender, target_size=100)
    if av_img:
        cv.create_image(cx_, cy_, image=av_img, anchor="center")
        cv._w_img = av_img
    else:
        cv.create_text(cx_, cy_, text="♂" if gender == "boy" else "♀",
                       fill=w_col, font=("Courier New", 44, "bold"))

    cv.create_text(cx_, cy_ + 76, text=winner.upper() + "  👑",
                   fill=w_col, font=("Courier New", 15, "bold"))

    lb = tk.Frame(screen, bg=BG_PANEL, highlightbackground=BORDER_VIVID, highlightthickness=1)
    cv.create_window(WIN_W//2, 468, window=lb, width=640, height=230)

    tk.Label(lb, text="FINAL  LEADERBOARD", fg=CYAN, bg=BG_PANEL,
             font=("Courier New", 12, "bold")).pack(pady=(8, 4))
    tk.Frame(lb, bg=BORDER_VIVID, height=1).pack(fill="x", padx=20, pady=(0, 6))

    medals   = ["🥇", "🥈", "🥉"]
    sorted_p = sorted(players, key=lambda x: scores[x], reverse=True)

    for rank, p in enumerate(sorted_p):
        col  = PLAYER_COLORS[players.index(p) % len(PLAYER_COLORS)]
        is_w = (p == winner)
        row  = tk.Frame(lb, bg="#1c1a00" if is_w else BG_PANEL)
        row.pack(fill="x", pady=3, padx=16)

        tk.Label(row, text=medals[rank] if rank < 3 else f"#{rank+1}",
                 fg=YELLOW, bg=row.cget("bg"),
                 font=("Courier New", 13), width=3).pack(side="left", padx=(0, 6))

        tk.Label(row, text=f"{p.upper()}" + ("  👑" if is_w else ""),
                 fg=col, bg=row.cget("bg"),
                 font=("Courier New", 12, "bold" if is_w else "normal")
                 ).pack(side="left", padx=4)

        tk.Label(row, text=f"{scores[p]}  pts",
                 fg=YELLOW if is_w else WHITE,
                 bg=row.cget("bg"),
                 font=("Courier New", 12, "bold")).pack(side="right")

    btn_row = tk.Frame(screen, bg=BG_DEEP)
    cv.create_window(WIN_W//2, WIN_H - 36, window=btn_row)

    def play_again():
        global players, player_avatars, scores
        players = []; player_avatars = {}; scores = {}
        show_screen("start")

    neon_btn(btn_row, "▶  PLAY AGAIN", GREEN, play_again, width=18).pack(side="left", padx=12)
    neon_btn(btn_row, "✕  QUIT",       RED,   main_window.destroy, width=12).pack(side="left", padx=12)

    return screen


# =============================================================
#  ENTRY POINT
# =============================================================
if __name__ == "__main__":
    initialize_main_window()
    show_screen("start")
    main_window.mainloop()